In [1]:
# !rm -rf /kaggle/working/Real-ESRGAN
# !git clone --depth 1 https://github.com/aksjfds/Real-ESRGAN.git /kaggle/working/Real-ESRGAN
%pip install -q -r requirements.txt
!python -m py_compile realesrgan.py realesrgan_fast.py realesrgan_fast_entry.py enhance/*.py
!python realesrgan_fast_entry.py --help >/dev/null


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
INPUT_VIDEO = "cm_4.mp4"
OUTPUT_VIDEO = "realesrgan_basicvsrpp.mp4"

MODEL = "realesr-animevideov3"
MODEL_PATH = ""
SCALE = 2
FPS = "source"

START_TIME = 3 * 60 + 15
TEST_SECONDS = 10
PROGRESS_INTERVAL = 60.0

# BasicVSR++ compressed-video enhancement runs before Real-ESRGAN.
BASICVSRPP = True
BASICVSRPP_TRACK = 1             # 1=fidelity; 2=perceptual; 3=fixed-bitrate fidelity
BASICVSRPP_MODEL_PATH = ""       # empty: download the official checkpoint
BASICVSRPP_GPU = 0               # primary GPU; tile inference also uses other visible GPUs
BASICVSRPP_FP16 = True
BASICVSRPP_CLIP_LENGTH = 9        # longer context; automatically falls back to previous 7 on OOM
BASICVSRPP_CLIP_OVERLAP = 2       # 9/2 gives temporal hop 5 instead of 3
BASICVSRPP_TILE_SIZE = 512        # quality baseline; runtime only selects tiles >=512
BASICVSRPP_TILE_PAD = 32
BASICVSRPP_STRENGTH = 1.0
BASICVSRPP_SCENE_THRESHOLD = 0.30

# Real-ESRGAN shared-memory auto tuning.
AUTO_TILE = True
MAX_TILE_SIZE = 1024              # quality-safe candidates are tried largest-first
AUTO_BATCH = True
MAX_BATCH_SIZE = 16               # probed independently on each GPU
TILE_SIZE = 256                   # fallback when auto tile is disabled
TILE_PAD = 10
TILE_VERIFY_COVERAGE = False      # debug-only check; disabling does not change pixels
BATCH_SIZE = 4                    # fallback when auto batch is disabled
GPU_IDS = "0,1"

COLOR_POLICY = "preserve"        # preserve / bt709
HDR_POLICY = "reject"            # reject / passthrough

VIDEO_CODEC = "hevc_nvenc"
OUTPUT_PIX_FMT = "auto"
CRF = 18
PRESET = "medium"
CQ = 18
NVENC_PRESET = "p7"
ENCODE_GPU = 0
AUDIO_CODEC = "copy"
AUDIO_BITRATE = "192k"


In [4]:
import shlex
import subprocess
import sys

command = [
    sys.executable, "realesrgan_fast_entry.py",
    "--input", INPUT_VIDEO, "--output", OUTPUT_VIDEO,
    "--model", MODEL, "--model-path", MODEL_PATH,
    "--scale", str(SCALE), "--fps", FPS,
    "--fp16", "--channels-last",
    "--basicvsrpp" if BASICVSRPP else "--no-basicvsrpp",
    "--basicvsrpp-track", str(BASICVSRPP_TRACK),
    "--basicvsrpp-model-path", BASICVSRPP_MODEL_PATH,
    "--basicvsrpp-gpu", str(BASICVSRPP_GPU),
    "--basicvsrpp-fp16" if BASICVSRPP_FP16 else "--no-basicvsrpp-fp16",
    "--basicvsrpp-clip-length", str(BASICVSRPP_CLIP_LENGTH),
    "--basicvsrpp-clip-overlap", str(BASICVSRPP_CLIP_OVERLAP),
    "--basicvsrpp-tile-size", str(BASICVSRPP_TILE_SIZE),
    "--basicvsrpp-tile-pad", str(BASICVSRPP_TILE_PAD),
    "--basicvsrpp-strength", str(BASICVSRPP_STRENGTH),
    "--basicvsrpp-scene-threshold", str(BASICVSRPP_SCENE_THRESHOLD),
    "--auto-tile" if AUTO_TILE else "--no-auto-tile",
    "--max-tile-size", str(MAX_TILE_SIZE),
    "--auto-batch" if AUTO_BATCH else "--no-auto-batch",
    "--max-batch-size", str(MAX_BATCH_SIZE),
    "--tile-size", str(TILE_SIZE), "--tile-pad", str(TILE_PAD),
    "--tile-verify-coverage" if TILE_VERIFY_COVERAGE else "--no-tile-verify-coverage",
    "--batch-size", str(BATCH_SIZE), "--gpu-ids", GPU_IDS,
    "--color-policy", COLOR_POLICY, "--hdr-policy", HDR_POLICY,
    "--video-codec", VIDEO_CODEC, "--output-pix-fmt", OUTPUT_PIX_FMT,
    "--crf", str(CRF), "--preset", PRESET, "--cq", str(CQ),
    "--nvenc-preset", NVENC_PRESET, "--encode-gpu", str(ENCODE_GPU),
    "--audio-codec", AUDIO_CODEC, "--audio-bitrate", AUDIO_BITRATE,
    "--start-time", str(START_TIME), "--test-seconds", str(TEST_SECONDS),
    "--progress-interval", str(PROGRESS_INTERVAL),
    "--ffmpeg-bin", "ffmpeg", "--ffprobe-bin", "ffprobe",
]
print("[command]", shlex.join(command), flush=True)
subprocess.run(command, check=True)


[command] /usr/local/bin/python realesrgan_fast_entry.py --input cm_4.mp4 --output realesrgan_basicvsrpp.mp4 --model realesr-animevideov3 --model-path '' --scale 2 --fps source --fp16 --channels-last --basicvsrpp --basicvsrpp-track 1 --basicvsrpp-model-path '' --basicvsrpp-gpu 0 --basicvsrpp-fp16 --basicvsrpp-clip-length 9 --basicvsrpp-clip-overlap 2 --basicvsrpp-tile-size 512 --basicvsrpp-tile-pad 32 --basicvsrpp-strength 1.0 --basicvsrpp-scene-threshold 0.3 --auto-tile --max-tile-size 1024 --auto-batch --max-batch-size 16 --tile-size 256 --tile-pad 10 --no-tile-verify-coverage --batch-size 4 --gpu-ids 0,1 --color-policy preserve --hdr-policy reject --video-codec hevc_nvenc --output-pix-fmt auto --crf 18 --preset medium --cq 18 --nvenc-preset p7 --encode-gpu 0 --audio-codec copy --audio-bitrate 192k --start-time 195 --test-seconds 10 --progress-interval 60.0 --ffmpeg-bin ffmpeg --ffprobe-bin ffprobe


Traceback (most recent call last):
  File "/code/Real-ESRGAN/realesrgan_fast_entry.py", line 115, in <module>
    fast.main()
  File "/code/Real-ESRGAN/realesrgan_fast.py", line 883, in main
    base.process_video(args)
  File "/code/Real-ESRGAN/realesrgan.py", line 1036, in process_video
    gpu_ids = parse_gpu_ids(args.gpu_ids)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/code/Real-ESRGAN/realesrgan.py", line 912, in parse_gpu_ids
    raise ValueError(f"Requested GPU {max(ids)}, but only {count} CUDA device(s) are visible.")
ValueError: Requested GPU 1, but only 1 CUDA device(s) are visible.


CalledProcessError: Command '['/usr/local/bin/python', 'realesrgan_fast_entry.py', '--input', 'cm_4.mp4', '--output', 'realesrgan_basicvsrpp.mp4', '--model', 'realesr-animevideov3', '--model-path', '', '--scale', '2', '--fps', 'source', '--fp16', '--channels-last', '--basicvsrpp', '--basicvsrpp-track', '1', '--basicvsrpp-model-path', '', '--basicvsrpp-gpu', '0', '--basicvsrpp-fp16', '--basicvsrpp-clip-length', '9', '--basicvsrpp-clip-overlap', '2', '--basicvsrpp-tile-size', '512', '--basicvsrpp-tile-pad', '32', '--basicvsrpp-strength', '1.0', '--basicvsrpp-scene-threshold', '0.3', '--auto-tile', '--max-tile-size', '1024', '--auto-batch', '--max-batch-size', '16', '--tile-size', '256', '--tile-pad', '10', '--no-tile-verify-coverage', '--batch-size', '4', '--gpu-ids', '0,1', '--color-policy', 'preserve', '--hdr-policy', 'reject', '--video-codec', 'hevc_nvenc', '--output-pix-fmt', 'auto', '--crf', '18', '--preset', 'medium', '--cq', '18', '--nvenc-preset', 'p7', '--encode-gpu', '0', '--au